# Notebook 17 — Residual neural network and temporal ensembles

**Development-only experimentation.** Compare the frozen Logistic L2 benchmark, Random Forest, a custom residual PyTorch MLP, fixed soft-voting blends, and temporal stacking. No final holdout reuse and no prospective ledger changes.

**Caution:** stacking uses chronological inner out-of-fold predictions, but inner preprocessing was fitted on the outer training set (including later inner dates). Thus its outer-fold evaluation remains separated from the outer test, while the inner meta-training process is not a fully nested preprocessing pipeline. Treat results as exploratory rather than production-grade stacking validation.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
if not (ROOT/'src'/'stock_ml_dynamic').exists():raise RuntimeError('Run from stock_ml_dynamic repository')
sys.path.insert(0,str(ROOT/'src'))
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.architecture_ensemble import run_architecture_ensemble
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():print('GPU:',torch.cuda.get_device_name(0))
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
candidates=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
master_dir=next((p for p in candidates if all((p/f).exists() for f in required)),None)
if master_dir is None:raise FileNotFoundError('Run Notebook 03 to create master table artifacts')
master,latest,bundle,manifest=load_master(master_dir)
print('Rows:',len(master),'symbols:',master.symbol.nunique())

## Run settings

For a quick smoke test use `N_FOLDS=2`, `NEURAL_EPOCHS=8`, and `INCLUDE_STACKING=False`. Restore five folds, 35 epochs, and stacking for full development evaluation. Expect significant CPU time for temporal stacking.

In [ ]:
N_FOLDS=5
NEURAL_EPOCHS=35
MAX_TRAIN_ROWS=60_000
TREE_TRAIN_ROWS=45_000
INCLUDE_STACKING=True
OUTPUT=ROOT/'artifacts'/'architecture_ensemble'/'1d'

scores,summary,eligibility,development_end=run_architecture_ensemble(master,n_folds=N_FOLDS,neural_epochs=NEURAL_EPOCHS,max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,include_stacking=INCLUDE_STACKING,output_dir=OUTPUT)
print('Development boundary:',development_end)
print('Final holdout NOT REUSED; prospective ledger NOT TOUCHED')

## Comparison — development CV only

In [ ]:
display(summary)
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(summary.model,summary.roc_auc)
ax.set_ylabel('Macro OVR ROC-AUC');ax.set_ylim(.45,max(.6,float(summary.roc_auc.max())+.02))
ax.tick_params(axis='x',rotation=35)
plt.tight_layout();plt.show()

## Fold variability and residual training history

In [ ]:
display(scores.pivot(index='fold',columns='model',values='roc_auc'))
if (OUTPUT/'residual_training_history.csv').exists():
    history=pd.read_csv(OUTPUT/'residual_training_history.csv')
    fig,ax=plt.subplots(figsize=(9,4))
    for fold,g in history.groupby('fold'):ax.plot(g.epoch,g.val_loss,label=f'Fold {fold}')
    ax.set(xlabel='Epoch',ylabel='Validation cross-entropy',title='Residual MLP chronological early stopping')
    ax.legend();plt.tight_layout();plt.show()

## Interpretation and next steps

- Compare **ROC-AUC first**, then balanced accuracy, log-loss, and fit time. Small differences may be sampling noise.
- Neural validation is a chronological subset of each outer training fold, not the outer test fold.
- Fixed voting weights are not optimized against the outer test fold.
- Temporal stacking is exploratory; for deployment, rebuild all feature preprocessing separately inside each inner fold.
- Do not reopen Notebook 13 holdout or alter the Notebook 15 prospective ledger to select a winner.
- This is predictive discrimination, **not** evidence of trading profitability.